# Proyecto 2

* Buscar datos del precio de las acciones
* Hacer análisis solicitados por el gestor
    * Cotización máxima
    * Cotización mínima
    * Valor medio
* Enviar correo electrónico


## Buscar datos del precio de las acciones

In [ ]:
%pip install yfinance
%pip install matplotlib 

In [ ]:
import yfinance

In [ ]:
tiker = input("Escriba el código de la acción: ")

data = yfinance.download([tiker], period="6mo")

cierre = data.Close[tiker]

In [ ]:
cierre.plot()

## Hacer los análisis solicitados por el gestor 

* Cotización máxima
* Cotización mínima
* Cotización media

In [ ]:
maxima = round(cierre.max(), 2)
minima = round(cierre.min(), 2)
valor_medio = round(cierre.mean(), 2)

In [ ]:
print(maxima)
print(minima)
print(valor_medio)

## Enviar correos electrónicos

In [ ]:
%pip install playwright
!playwright install chromium

In [ ]:

import webbrowser
import time 

In [ ]:

destinatario = "edig0rgudevia@gmail.com"
asunto = f"Análisis acciones {tiker} últimos 6 meses"
mensaje = f'''
Buenas noches Zai,

Acá te envío el análisis de las acciones de Apple de los últimos 6 meses:

Cotización máxima: USD {maxima}
Cotización mínima: USD {minima}
Valor medio: USD {valor_medio}

¡Estoy pendiente a cualquier observación!

Atentamente, 
Diego Rodríguez
'''





In [ ]:
import os
from playwright.async_api import async_playwright

# Usa un nombre completamente nuevo para evitar bloqueos con los procesos anteriores
user_data_dir = os.path.abspath("./gmail_session_v2")

async with async_playwright() as p:
    context = await p.chromium.launch_persistent_context(
        user_data_dir,
        channel="chrome",
        headless=False,
        args=[
            "--start-maximized",
            "--disable-blink-features=AutomationControlled"
        ],
        ignore_default_args=["--enable-automation"]
    )
    
    page = context.pages[0] if context.pages else await context.new_page()
    await page.goto("https://mail.google.com/")
    
    print("Ventana de Chrome abierta.")
    print("Inicia sesión en tu cuenta de Gmail normalmente.")
    input("Cuando ya estés viendo tu bandeja de entrada, presiona ENTER aquí...")
    
    await context.close()

print("¡Nueva sesión guardada exitosamente!")

In [ ]:
import os
from playwright.async_api import async_playwright

user_data_dir = os.path.abspath("./gmail_session_v2")

async with async_playwright() as p:
    context = await p.chromium.launch_persistent_context(
        user_data_dir,
        channel="chrome",
        headless=False,
        # Pausa de 1 segundo (1000 ms) entre cada acción de Playwright
        slow_mo=1000,
        args=[
            "--start-maximized",
            "--disable-blink-features=AutomationControlled"
        ],
        ignore_default_args=["--enable-automation"]
    )
    
    page = context.pages[0] if context.pages else await context.new_page()
    await page.goto("https://mail.google.com/", wait_until="domcontentloaded")
    
    # Pausa de cortesía para que la interfaz termine de renderizar
    await page.wait_for_timeout(2000)

    # 1. Pulsar Redactar
    redactar = page.locator('div[role="button"]:has-text("Redactar"), div[role="button"]:has-text("Compose"), div[gh="cm"]').first
    await redactar.wait_for(state="visible", timeout=30000)
    await redactar.click()
    await page.wait_for_timeout(1500)

    # 2. Destinatario (escribe letra por letra con pausa de 50ms)
    destinatario_input = page.locator('input[aria-label*="Para"], input[aria-label*="To"], input[peoplekit-id]').first
    await destinatario_input.wait_for(state="visible", timeout=10000)
    await destinatario_input.press_sequentially(destinatario, delay=50)
    await destinatario_input.press("Enter")
    await page.wait_for_timeout(1000)

    # 3. Asunto
    asunto_input = page.locator('input[name="subjectbox"]')
    await asunto_input.wait_for(state="visible", timeout=5000)
    await asunto_input.press_sequentially(asunto, delay=40)
    await page.wait_for_timeout(1000)

    # 4. Cuerpo del mensaje
    cuerpo = page.locator('div[aria-label*="Cuerpo del mensaje"], div[aria-label*="Message Body"]').first
    await cuerpo.wait_for(state="visible", timeout=5000)
    await cuerpo.press_sequentially(mensaje, delay=20)
    
    # Pausa antes de despachar el correo para revisión visual
    await page.wait_for_timeout(2000)

    # 5. Enviar usando atajo nativo
    await page.keyboard.press("Control+Enter")

    # Espera para asegurar la confirmación de envío
    await page.wait_for_timeout(5000)
    await context.close()

print("¡Correo enviado a ritmo controlado!")